In [1]:
import numpy as np
import pandas as pd
from astropy.io import fits
import matplotlib.pyplot as plt
import lightkurve as lk
import glob
import re
from astroquery import vizier
import pandas as pd
from astroquery.mast import Catalogs
from astropy.table import Table

In [3]:
final_tic_converted = ['257436461', '435339847', '337607207', '39936432', '392469577', '266014079', '14570099', '179032688', '376938120', '187309502', '333605244', '366631954', '307733361', '332022997', '238608004', '376939759', '418761354', '293460551', '12822545', '187960878', '440687723', '175241633', '7059054', '297146957', '38258419', '2621212', '175265494', '293375721', '175194958', '380884458', '187273811', '175290532', '26078330', '195193025', '14635562', '677945', '115072772', '12181371', '741596', '146799150', '90090343', '294319820', '458686847', '423358488', '125060509', '175262071', '113979956', '376936928', '135043332', '14111411', '2521105', '332024125', '434226736', '11023038', '15756231', '12770870', '86130229', '197246627', '422349422', '441907126', '10195089', '335102224',  '396972464', '184914317', '49652731', '301258470', '68504570', '446672575', '291090078', '49040478', '4610830', '428820090', '281731203', '301610040', '175193677', '363445338', '335322931', '363445121', '27039476', '301235044', '323687123', '345063080', '380619414', '436575927', '38064757', '277869696', '397022282', '38087018', '18310799', '281886199', '5882269', '150096001', '456945304', '49462779', '50171060', '388804061', '422487869', '82050863', '363444743', '173090902', '443616612', '392817207', '432247315', '903075188', '61312205']


final_periods = [33.122, 20.686, 8.733, 29.429, 17.988, 9.007, 30.178, 24.824, 33.119, 20.866, 26.33, 24.244, 4.263, 9.117, 18.253, 40.0, 33.122, 30.19, 16.414, 16.826, 23.532, 11.113, 10.187, 35.639, 21.761, 15.821, 10.788, 16.23, 17.125, 30.5, 13.625, 40.0, 8.891, 24.244, 33.181, 12.03, 26.855, 26.812, 29.005, 12.677, 23.688, 23.065, 9.179, 14.466, 13.047, 11.457, 35.822, 40.0, 7.056, 25.266, 17.997, 20.239, 1.878, 17.552, 2.844, 17.263, 9.557, 36.182, 28.418, 14.436, 40.0, 40.0, 15.393, 21.954, 23.218, 12.356, 12.226, 7.849, 14.823, 6.28, 12.356, 9.655, 18.794, 11.428, 8.33, 7.698, 15.016, 8.517, 18.132, 7.044, 4.338, 14.053, 6.562, 28.531, 13.341, 7.999, 21.062, 21.062, 14.768, 14.294, 8.854, 36.347, 13.289, 19.057, 17.077, 36.367, 23.366, 36.367, 14.294, 20.009, 9.763, 6.637, 10.868, 21.062, 36.279]

In [4]:
def get_coords_from_tic(tic_ids):
    """
    Queries MAST to get RA and Dec for a list of TIC IDs.
    """
    # Ensure IDs are strings for the query
    tic_ids = [str(tid) for tid in tic_ids]
    
    print(f"Querying MAST for {len(tic_ids)} IDs...")
    
    # Query the TIC catalog
    # 'ID' in query_criteria refers to the TIC ID
    result_table = Catalogs.query_criteria(catalog="TIC", ID=tic_ids)
    
    # Convert to pandas for easier manipulation
    df_coords = result_table.to_pandas()
    
    # Select only the essential columns
    # 'ra' and 'dec' are J2000 coordinates
    df_coords = df_coords[['ID', 'ra', 'dec']]
    df_coords.columns = ['TIC_ID', 'RA', 'Dec']
    
    return df_coords

coord_df = get_coords_from_tic(final_tic_converted)

print("\nResults:")
print(coord_df)

# To use these in your existing cross-match dictionary:
# ra_dict = dict(zip(coord_df['TIC_ID'].astype(int), coord_df['RA']))
# dec_dict = dict(zip(coord_df['TIC_ID'].astype(int), coord_df['Dec']))

Querying MAST for 105 IDs...

Results:
        TIC_ID          RA        Dec
0    125060509  293.560726 -23.131101
1    175194958  129.636817  19.773775
2    293375721  131.266611  13.549830
3    175193677  129.689668  23.685906
4     10195089  288.026922 -21.007641
..         ...         ...        ...
100   18310799   67.412474  22.882721
101  436575927   71.109826  16.518680
102   14570099  125.987074  22.663038
103  363444743  169.483616   5.988393
104  293460551  132.009733  16.901863

[105 rows x 3 columns]


In [5]:
from astropy.table import Table

jayasinghe = Table.read('/Users/iansterrett/Downloads/jayasinghe2018/catv2021.dat', readme='https://cdsarc.cds.unistra.fr/ftp/II/366/ReadMe', format='ascii.cds').to_pandas()
print('catalogue read in done')

mega_list = pd.read_csv('ASTR502_Mega_Target_List.csv')
print('csv read in done')



catalogue read in done
csv read in done


In [8]:
mega_list['tic_id'] = mega_list['tic_id'].str.replace('TIC ', '', regex=False).astype('Int64')

In [10]:
mega_with_jayaisinghe = mega_list.merge(jayasinghe[['TIC', 'Per']], left_on='tic_id', right_on='TIC').drop_duplicates(subset='tic_id')

In [13]:
print(mega_with_jayaisinghe)

             pl_name     hostname                   gaia_dr3_id  \
0           KELT-1 b       KELT-1  Gaia DR3 2881784280929695360   
1         TOI-1224 c     TOI-1224  Gaia DR3 4620009665047355520   
3         TOI-6109 b     TOI-6109   Gaia DR3 241035596174886016   
5            K2-89 b        K2-89    Gaia DR3 63206117414110848   
6        V1298 Tau b    V1298 Tau    Gaia DR3 51886335968692480   
10       TOI-3353.01     TOI-3353  Gaia DR3 5212899427468919296   
11         TOI-500 b      TOI-500  Gaia DR3 5509620021956148736   
12       TOI-5082.01     TOI-5082  Gaia DR3 3368214650329888512   
13        HD 63433 b     HD 63433   Gaia DR3 875071278432954240   
16        HD 73583 b     HD 73583  Gaia DR3 5746824674801810816   
18          G 9-40 b       G 9-40   Gaia DR3 684992690384102528   
19       HIP 67522 b    HIP 67522  Gaia DR3 6113920619134019456   
21        TOI-1860 b     TOI-1860  Gaia DR3 1620024491110370688   
22          K2-240 c       K2-240  Gaia DR3 625762571943098201

In [12]:
print(len(mega_with_jayaisinghe))

35


In [14]:
jayasinghe = Table.read('/Users/iansterrett/Downloads/long2023/table5..dat', readme='https://cdsarc.cds.unistra.fr/ftp/J/ApJS/268/30/ReadMe', format='ascii.cds').to_pandas()
print('catalogue read in done')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/iansterrett/Downloads/long2023/table5..dat'